# 🔦 Day 22 — Sparse Autoencoders, Uncertainty Estimation & Selective Prediction
**Duration:** 6 Hours | **Total run time: under 60 seconds**

| Session | Time | Topic | Run Time |
|---------|------|-------|----------|
| 1 | 9:00–10:30 AM | Sparse Autoencoders & Mechanistic Interpretability | ~11 sec |
| 2 | 10:30–11:00 AM | MC Dropout Uncertainty Estimation | ~5 sec |
| 3 | 11:00 AM–1:00 PM | Deep Ensembles + Selective Prediction | ~20 sec |
| 4 | 1:00–2:00 PM | Lunch Break | — |
| 5 | 2:00–4:00 PM | Research Capstone: Interpretable Uncertain ML | ~18 sec |

> **Zero downloads. Pure numpy + sklearn.**
---

In [ ]:
# !pip install scikit-learn matplotlib pandas numpy scipy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import warnings, time
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer, load_wine, load_iris, make_classification
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
import sklearn
print('All imports OK')
print(f'   sklearn: {sklearn.__version__}')

---
## Session 1 — Sparse Autoencoders & Mechanistic Interpretability
**Time:** 9:00–10:30 AM | **Run time: ~11 sec**

In [ ]:
# 1.1  Sparse Autoencoder with L1 Penalty
class SparseAutoencoder:
    """
    Sparse Autoencoder: x → z → x_hat
    Loss = ||x - x_hat||^2 + λ||z||_1
    ReLU encoder (non-negative activations)
    L1 sparsity forces most z_i = 0 at any given input
    """
    def __init__(self, d_in, d_latent, sparsity=0.05, lr=0.005, seed=42):
        rng = np.random.default_rng(seed)
        # Encoder and decoder weight matrices
        self.W_enc = rng.normal(0, 0.1, (d_in, d_latent))
        self.b_enc = np.zeros(d_latent)
        self.W_dec = rng.normal(0, 0.1, (d_latent, d_in))
        self.b_dec = np.zeros(d_in)
        self.lam   = sparsity
        self.lr    = lr
        self.losses = []
        self.recon_losses = []
        self.l1_losses    = []

    def encode(self, X):
        return np.maximum(0, X @ self.W_enc + self.b_enc)  # ReLU

    def decode(self, Z):
        return Z @ self.W_dec + self.b_dec

    def forward(self, X):
        Z     = self.encode(X)
        X_hat = self.decode(Z)
        return Z, X_hat

    def step(self, X):
        Z, X_hat = self.forward(X)
        n = len(X)
        # Losses
        recon_loss = np.mean((X - X_hat)**2)
        l1_loss    = self.lam * np.mean(np.abs(Z))
        # Gradients
        dL_dXhat = 2 * (X_hat - X) / n
        # Decoder gradients
        dL_dWdec = Z.T @ dL_dXhat
        dL_dbdec = dL_dXhat.sum(axis=0)
        # Encoder gradients (through ReLU)
        dL_dZ    = dL_dXhat @ self.W_dec.T * (Z > 0)  # ReLU derivative
        dL_dZ   += self.lam * np.sign(Z) / n           # L1 subgradient
        dL_dWenc = X.T @ dL_dZ
        dL_dbenc = dL_dZ.sum(axis=0)
        # Update
        self.W_dec -= self.lr * dL_dWdec
        self.b_dec -= self.lr * dL_dbdec
        self.W_enc -= self.lr * dL_dWenc
        self.b_enc -= self.lr * dL_dbenc
        self.losses.append(recon_loss + l1_loss)
        self.recon_losses.append(recon_loss)
        self.l1_losses.append(l1_loss)
        return recon_loss, l1_loss

# Train SAE
rng = np.random.default_rng(42)
N_TRAIN, D_IN, D_LATENT = 800, 20, 80
X_sae = rng.normal(0, 1, (N_TRAIN, D_IN))

sae = SparseAutoencoder(d_in=D_IN, d_latent=D_LATENT, sparsity=0.05, lr=0.008)
for step in range(300):
    # Mini-batch
    idx   = rng.integers(0, N_TRAIN, 100)
    sae.step(X_sae[idx])

Z_all   = sae.encode(X_sae)
X_recon = sae.decode(Z_all)
recon_mse = np.mean((X_sae - X_recon)**2)
n_dead    = ((Z_all > 0).mean(axis=0) < 0.01).sum()
avg_active = (Z_all > 0).mean()

print(f'SAE Training Results:')
print(f'  Input dim     : {D_IN}  →  Latent dim: {D_LATENT}')
print(f'  Reconstruction MSE: {recon_mse:.6f}')
print(f'  Dead latents  : {n_dead}/{D_LATENT}  ({n_dead/D_LATENT:.1%})')
print(f'  Avg active    : {avg_active:.1%}  (sparsity achieved)')

In [ ]:
# 1.2  TopK Activation (Avoids Dead Latents)
class TopKSparseAutoencoder:
    """
    Instead of L1 penalty, keep exactly top-K activations per sample.
    Guarantees sparsity without dead latents.
    """
    def __init__(self, d_in, d_latent, k=8, lr=0.005, seed=42):
        rng = np.random.default_rng(seed)
        self.W_enc = rng.normal(0, 0.1, (d_in, d_latent))
        self.W_dec = rng.normal(0, 0.1, (d_latent, d_in))
        self.k     = k
        self.lr    = lr
        self.losses = []

    def topk_encode(self, X):
        H = np.maximum(0, X @ self.W_enc)       # pre-activation
        Z = np.zeros_like(H)
        topk_idx = np.argsort(H, axis=1)[:, -self.k:]  # top-k per row
        for i, idx in enumerate(topk_idx):
            Z[i, idx] = H[i, idx]
        return Z

    def step(self, X):
        Z     = self.topk_encode(X)
        X_hat = Z @ self.W_dec
        loss  = np.mean((X - X_hat)**2)
        n     = len(X)
        dLdXhat = 2*(X_hat - X)/n
        self.W_dec -= self.lr * Z.T @ dLdXhat
        dLdZ = dLdXhat @ self.W_dec.T * (Z > 0)
        self.W_enc -= self.lr * X.T @ dLdZ
        self.losses.append(loss)
        return loss

topk_sae = TopKSparseAutoencoder(d_in=D_IN, d_latent=D_LATENT, k=8)
for step in range(300):
    idx = rng.integers(0, N_TRAIN, 100)
    topk_sae.step(X_sae[idx])

Z_topk   = topk_sae.topk_encode(X_sae)
n_dead_topk  = ((Z_topk > 0).mean(axis=0) < 0.01).sum()
avg_act_topk = (Z_topk > 0).mean()
recon_topk   = np.mean((X_sae - Z_topk @ topk_sae.W_dec)**2)

print(f'\nTopK-SAE (k={topk_sae.k}):')
print(f'  Reconstruction MSE: {recon_topk:.6f}')
print(f'  Dead latents  : {n_dead_topk}/{D_LATENT}  (should be 0 with TopK)')
print(f'  Avg active    : {avg_act_topk:.1%}  (exactly k/{D_LATENT} = {topk_sae.k/D_LATENT:.1%})')

In [ ]:
# 1.3  Feature Dictionary Analysis
# Train SAE on real model activations (RF leaf node activations as proxy)
X_real, y_real = load_breast_cancer(return_X_y=True)
scaler_sae = StandardScaler().fit(X_real)
X_sc_sae   = scaler_sae.transform(X_real)

# Use RF to generate 'activation' features
rf_feat = RandomForestClassifier(50, random_state=42).fit(X_sc_sae, y_real)
# Use predict_proba per tree as proxy activations (50 trees × 2 classes)
tree_probs = np.array([tree.predict_proba(X_sc_sae)[:,1]
                       for tree in rf_feat.estimators_]).T  # (n, 50)

# Train SAE on tree activations
sae_feat = SparseAutoencoder(d_in=50, d_latent=100, sparsity=0.08, lr=0.01)
for _ in range(200):
    idx = rng.integers(0, len(tree_probs), 64)
    sae_feat.step(tree_probs[idx])

Z_feat = sae_feat.encode(tree_probs)
# Most active latent features overall
top_features = np.argsort(Z_feat.mean(axis=0))[::-1][:10]
print('Top 10 most active SAE feature directions:')
for rank, feat_idx in enumerate(top_features, 1):
    activation_rate = (Z_feat[:, feat_idx] > 0).mean()
    mean_val        = Z_feat[:, feat_idx][Z_feat[:, feat_idx] > 0].mean() if activation_rate > 0 else 0
    print(f'  Feature {feat_idx:3d}: activation_rate={activation_rate:.2%}  mean_when_active={mean_val:.4f}')

In [ ]:
# 1.4  SAE Visualisation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Loss curves
axes[0].plot(sae.recon_losses, color='#534AB7', linewidth=1.5, label='Reconstruction')
axes[0].plot(sae.l1_losses,    color='#D85A30', linewidth=1.5, label='L1 sparsity')
axes[0].set_title('SAE Training Loss')
axes[0].set_xlabel('Step'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Activation frequency histogram
act_freq = (Z_all > 0).mean(axis=0)
axes[1].hist(act_freq, bins=20, color='#1D9E75', alpha=0.85, edgecolor='white')
axes[1].axvline(0.01, color='#D85A30', linestyle='--', linewidth=2, label='Dead threshold (1%)')
axes[1].set_title('Latent Activation Frequency')
axes[1].set_xlabel('Fraction of inputs that activate'); axes[1].legend()

# Feature activation heatmap (top features × classes)
for cls in [0, 1]:
    mask    = y_real == cls
    mean_act = Z_feat[mask].mean(axis=0)
    axes[2].plot(mean_act[top_features], label=f'Class {cls}', linewidth=1.8,
                 color=['#534AB7','#D85A30'][cls])
axes[2].set_title('SAE Feature Activation by Class')
axes[2].set_xlabel('Top feature index'); axes[2].set_ylabel('Mean activation')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Sparse Autoencoder Analysis', fontsize=12)
plt.tight_layout(); plt.show()

---
## Session 2 — MC Dropout Uncertainty Estimation
**Time:** 10:30–11:00 AM | **Run time: ~5 sec**

In [ ]:
# 2.1  MC Dropout on a Numpy Neural Network
class MCDropoutNet:
    """Small 2-layer net supporting MC Dropout inference."""
    def __init__(self, d_in, d_h, d_out=1, seed=42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 0.1, (d_in, d_h))
        self.b1 = np.zeros(d_h)
        self.W2 = rng.normal(0, 0.1, (d_h, d_out))
        self.b2 = np.zeros(d_out)

    def forward(self, X, dropout_rate=0.0, training=False):
        h = np.tanh(X @ self.W1 + self.b1)
        if dropout_rate > 0:
            # Apply dropout (scale by 1/(1-p) for inverted dropout)
            mask = np.random.binomial(1, 1-dropout_rate, h.shape)
            h    = h * mask / (1 - dropout_rate)
        logit = h @ self.W2 + self.b2
        return 1 / (1 + np.exp(-logit.squeeze()))  # sigmoid

    def mc_predict(self, X, T=50, dropout_rate=0.2):
        """T stochastic forward passes → mean + variance."""
        samples = np.array([self.forward(X, dropout_rate=dropout_rate) for _ in range(T)])
        return samples.mean(axis=0), samples.var(axis=0), samples

# Quick train on breast cancer
X_bc, y_bc = load_breast_cancer(return_X_y=True)
X_tr_bc, X_te_bc, y_tr_bc, y_te_bc = train_test_split(X_bc, y_bc, test_size=0.3, random_state=42)
sc = StandardScaler().fit(X_tr_bc)
Xtr_sc = sc.transform(X_tr_bc)
Xte_sc = sc.transform(X_te_bc)

net = MCDropoutNet(d_in=X_bc.shape[1], d_h=32)
# Simple gradient training
for epoch in range(100):
    rng_t = np.random.default_rng(epoch)
    idx   = rng_t.integers(0, len(Xtr_sc), 64)
    X_b   = Xtr_sc[idx]; y_b = y_tr_bc[idx].astype(float)
    # Forward
    prob  = net.forward(X_b, dropout_rate=0.2)
    # Gradient (BCE)
    err   = prob - y_b
    h     = np.tanh(X_b @ net.W1 + net.b1)
    net.W2 -= 0.01 * h.T @ err[:, None] / len(y_b)
    dh     = (err[:, None] @ net.W2.T) * (1 - h**2)
    net.W1 -= 0.01 * X_b.T @ dh / len(y_b)

# MC Dropout inference
mean_mc, var_mc, samples_mc = net.mc_predict(Xte_sc, T=100, dropout_rate=0.3)
preds_mc = (mean_mc > 0.5).astype(int)
acc_mc   = accuracy_score(y_te_bc, preds_mc)
print(f'MC Dropout ({100} samples):')
print(f'  Accuracy   : {acc_mc:.4f}')
print(f'  Mean unc   : {var_mc.mean():.6f}')
print(f'  High-unc % : {(var_mc > var_mc.mean()).mean():.1%}')

In [ ]:
# 2.2  Epistemic vs Aleatoric Uncertainty Decomposition
# Total uncertainty = epistemic + aleatoric
# Epistemic = variance of means across MC samples
# Aleatoric = mean of variances across MC samples

def decompose_uncertainty(samples):
    """
    samples: (T, n) — T MC predictions for n test points
    Returns: epistemic and aleatoric uncertainty per point
    """
    mean_pred  = samples.mean(axis=0)         # (n,)
    epistemic  = samples.var(axis=0)          # variance of means
    aleatoric  = (samples * (1-samples)).mean(axis=0)  # mean of Bernoulli variance
    total      = epistemic + aleatoric
    return epistemic, aleatoric, total

epi, alea, total = decompose_uncertainty(samples_mc)

print('Uncertainty Decomposition:')
print(f'  Epistemic (reducible)   : mean={epi.mean():.5f}  max={epi.max():.5f}')
print(f'  Aleatoric (irreducible) : mean={alea.mean():.5f}  max={alea.max():.5f}')
print(f'  Total                   : mean={total.mean():.5f}')

# Does uncertainty correlate with prediction errors?
correct = (preds_mc == y_te_bc)
print(f'\nMean uncertainty when CORRECT : {total[correct].mean():.5f}')
print(f'Mean uncertainty when WRONG   : {total[~correct].mean():.5f}')
print(f'Uncertainty separation (>0 = good): {total[~correct].mean()-total[correct].mean():.5f}')

---
## Session 3 — Deep Ensembles + Selective Prediction
**Time:** 11:00 AM–1:00 PM | **Run time: ~20 sec**

In [ ]:
# 3.1  Deep Ensemble
class DeepEnsemble:
    """
    Train M diverse models with different random seeds.
    Aggregate predictions → mean (prediction) + variance (uncertainty).
    Diversity comes from different initialisations and data ordering.
    """
    def __init__(self, n_members=5, model_class=RandomForestClassifier, **kwargs):
        self.models = [model_class(random_state=i, **kwargs)
                       for i in range(n_members)]
        self.n      = n_members

    def fit(self, X, y):
        for m in self.models:
            m.fit(X, y)
        return self

    def predict_proba(self, X):
        """Return (mean_prob, variance) across ensemble members."""
        probs = np.array([m.predict_proba(X) for m in self.models])  # (M, n, K)
        return probs.mean(axis=0), probs.var(axis=0)

    def predict(self, X):
        mean_p, _ = self.predict_proba(X)
        return mean_p.argmax(axis=1)

    def uncertainty(self, X):
        """Scalar uncertainty per sample = mean variance across classes."""
        _, var = self.predict_proba(X)
        return var.mean(axis=1)

X_ens, y_ens = load_breast_cancer(return_X_y=True)
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_ens, y_ens, test_size=0.25, random_state=42)

ens_model = DeepEnsemble(n_members=7, model_class=RandomForestClassifier,
                          n_estimators=50)
ens_model.fit(Xtr_e, ytr_e)

mean_p_e, var_p_e = ens_model.predict_proba(Xte_e)
preds_e           = ens_model.predict(Xte_e)
unc_e             = ens_model.uncertainty(Xte_e)
acc_e             = accuracy_score(yte_e, preds_e)

print(f'Deep Ensemble (M=7 RF):')
print(f'  Overall accuracy   : {acc_e:.4f}')
print(f'  Mean uncertainty   : {unc_e.mean():.6f}')
print(f'  High uncertainty % : {(unc_e > unc_e.mean()).mean():.1%}')

In [ ]:
# 3.2  Selective Prediction: Abstain Policy
class SelectivePredictor:
    """
    Abstain on samples where uncertainty exceeds threshold.
    Trade off coverage (fraction answered) for accuracy (on answered).
    """
    def __init__(self, ensemble, abstain_threshold=None):
        self.ensemble  = ensemble
        self.threshold = abstain_threshold

    def predict(self, X, threshold=None):
        thr   = threshold if threshold is not None else self.threshold
        unc   = self.ensemble.uncertainty(X)
        preds = self.ensemble.predict(X)
        mask  = unc <= thr  # True = answer, False = abstain
        return preds, mask, unc

    def coverage_accuracy_curve(self, X, y, n_thresholds=30):
        unc   = self.ensemble.uncertainty(X)
        preds = self.ensemble.predict(X)
        # Sort by uncertainty ascending
        order      = np.argsort(unc)
        unc_sorted = unc[order]
        thresholds = np.percentile(unc, np.linspace(10, 100, n_thresholds))
        results    = []
        for thr in thresholds:
            covered = unc <= thr
            if covered.sum() == 0:
                continue
            cov = covered.mean()
            acc = accuracy_score(y[covered], preds[covered])
            results.append({'threshold': thr, 'coverage': cov, 'accuracy': acc})
        return pd.DataFrame(results)

sp = SelectivePredictor(ens_model)
curve_df = sp.coverage_accuracy_curve(Xte_e, yte_e)

print('Coverage-Accuracy Trade-off:')
print(f'{"Threshold":12s} | {"Coverage":10s} | {"Accuracy"}')
print('-' * 38)
for _, row in curve_df.iloc[::5].iterrows():
    print(f'{row.threshold:12.6f} | {row.coverage:10.1%} | {row.accuracy:.4f}')

In [ ]:
# 3.3  Coverage-Accuracy Curve Visualisation
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Coverage-Accuracy curve
axes[0].plot(curve_df['coverage'], curve_df['accuracy'],
             'o-', color='#534AB7', linewidth=2, markersize=5)
axes[0].axhline(acc_e, linestyle='--', color='#D85A30',
                label=f'Full coverage acc={acc_e:.4f}')
axes[0].set_xlabel('Coverage (fraction answered)')
axes[0].set_ylabel('Accuracy on answered subset')
axes[0].set_title('Selective Prediction: Coverage vs Accuracy')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

# Uncertainty histogram: correct vs wrong
correct_e = (preds_e == yte_e)
axes[1].hist(unc_e[correct_e],  bins=20, alpha=0.6, color='#1D9E75',
             density=True, label='Correct')
axes[1].hist(unc_e[~correct_e], bins=10, alpha=0.7, color='#D85A30',
             density=True, label='Wrong')
axes[1].set_xlabel('Ensemble uncertainty')
axes[1].set_ylabel('Density')
axes[1].set_title('Uncertainty Distribution: Correct vs Wrong')
axes[1].legend(); axes[1].grid(alpha=0.3)

# SAE loss comparison
axes[2].plot(sae.losses,      color='#534AB7', linewidth=1.5, label='L1-SAE')
axes[2].plot(topk_sae.losses, color='#1D9E75', linewidth=1.5, label='TopK-SAE')
axes[2].set_title('SAE Loss: L1 vs TopK')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Loss')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

In [ ]:
# 3.4  MC Dropout vs Deep Ensemble Comparison
print('MC Dropout vs Deep Ensemble Comparison:')
print(f'{"Method":20s} | {"Accuracy":9s} | {"Mean Unc":10s} | {"Unc Separation"}')
print('-' * 65)

# MC Dropout stats
mcd_sep = total[~correct].mean() - total[correct].mean()
print(f'{"MC Dropout":20s} | {acc_mc:9.4f} | {total.mean():10.6f} | {mcd_sep:+.6f}')

# Deep ensemble stats
ens_sep = unc_e[~correct_e].mean() - unc_e[correct_e].mean()
print(f'{"Deep Ensemble":20s} | {acc_e:9.4f} | {unc_e.mean():10.6f} | {ens_sep:+.6f}')

print('\nKey insight: both methods assign higher uncertainty to wrong predictions.')
print('Deep ensembles are more reliable but require M full model evaluations.')

---
## Lunch Break — 1:00–2:00 PM
1. Why do dead latents indicate over-regularisation in L1-SAE?
2. How does MC Dropout approximate Bayesian posterior inference?
3. What is the difference between epistemic and aleatoric uncertainty?
4. Why does selective prediction always improve accuracy at the cost of coverage?
---

## Session 5 — Research Capstone: Interpretable Uncertain ML System
**Time:** 2:00–4:00 PM | **Run time: ~18 sec**

In [ ]:
# 5.1  Build the Full System
print('='*62)
print(' RESEARCH CAPSTONE: INTERPRETABLE UNCERTAIN ML SYSTEM')
print('='*62)

X_cap, y_cap = load_wine(return_X_y=True)
Xtr_cap, Xte_cap, ytr_cap, yte_cap = train_test_split(
    X_cap, y_cap, test_size=0.25, random_state=42
)
sc_cap = StandardScaler().fit(Xtr_cap)
Xtr_sc_cap = sc_cap.transform(Xtr_cap)
Xte_sc_cap = sc_cap.transform(Xte_cap)

print(f'Dataset: Wine  {X_cap.shape}  Train:{len(Xtr_cap)}  Test:{len(Xte_cap)}')

In [ ]:
# Stage 1: Train Deep Ensemble
print('\nStage 1 — Deep Ensemble (7 members)')
ens_cap = DeepEnsemble(n_members=7, model_class=GradientBoostingClassifier,
                        n_estimators=60, learning_rate=0.1, max_depth=3)
ens_cap.fit(Xtr_sc_cap, ytr_cap)

mean_cap, var_cap = ens_cap.predict_proba(Xte_sc_cap)
preds_cap         = ens_cap.predict(Xte_sc_cap)
unc_cap           = ens_cap.uncertainty(Xte_sc_cap)
acc_cap           = accuracy_score(yte_cap, preds_cap)

print(f'  Accuracy     : {acc_cap:.4f}')
print(f'  Mean unc     : {unc_cap.mean():.6f}')

In [ ]:
# Stage 2: Train SAE on Ensemble Activations
print('\nStage 2 — Sparse Autoencoder on Ensemble Activations')
# Use per-member class probabilities as 'activations'
all_probs   = np.array([m.predict_proba(Xtr_sc_cap) for m in ens_cap.models])
# Shape: (M, n_tr, n_classes) → flatten to (n_tr, M*n_classes)
activations  = all_probs.transpose(1,0,2).reshape(len(Xtr_sc_cap), -1)

sae_cap = SparseAutoencoder(
    d_in    = activations.shape[1],
    d_latent= 32,
    sparsity= 0.05,
    lr      = 0.01
)
for _ in range(200):
    idx = rng.integers(0, len(activations), 64)
    sae_cap.step(activations[idx])

# Encode test activations
test_probs_all = np.array([m.predict_proba(Xte_sc_cap) for m in ens_cap.models])
test_acts      = test_probs_all.transpose(1,0,2).reshape(len(Xte_sc_cap), -1)
Z_cap          = sae_cap.encode(test_acts)

n_dead_cap = ((sae_cap.encode(activations) > 0).mean(axis=0) < 0.01).sum()
print(f'  SAE latent dim   : {sae_cap.W_enc.shape[1]}')
print(f'  Dead latents     : {n_dead_cap}/{sae_cap.W_enc.shape[1]}')
print(f'  Avg active       : {(Z_cap > 0).mean():.1%}')

In [ ]:
# Stage 3: Selective Prediction with SAE-Enhanced Uncertainty
print('\nStage 3 — Selective Prediction')
sp_cap  = SelectivePredictor(ens_cap)
curve_cap = sp_cap.coverage_accuracy_curve(Xte_sc_cap, yte_cap, n_thresholds=20)

print(f'Coverage-Accuracy Curve (selective on uncertainty):')
for _, row in curve_cap.iloc[::4].iterrows():
    answered = int(row['coverage'] * len(yte_cap))
    print(f'  Coverage={row["coverage"]:.0%} ({answered}/{len(yte_cap)} answered)  '
          f'Accuracy={row["accuracy"]:.4f}')

# Best operating point: highest accuracy with > 60% coverage
filtered = curve_cap[curve_cap['coverage'] >= 0.6]
best_row  = filtered.loc[filtered['accuracy'].idxmax()]
print(f'\nBest operating point (coverage ≥ 60%):')
print(f'  Coverage = {best_row["coverage"]:.1%}')
print(f'  Accuracy = {best_row["accuracy"]:.4f}  (vs {acc_cap:.4f} without abstaining)')

In [ ]:
# Stage 4: Final Research Report + Visualisation
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. SAE latent activation heatmap
Z_cap_tr = sae_cap.encode(activations)
act_by_class = [Z_cap_tr[ytr_cap==c].mean(axis=0) for c in range(3)]
hm = np.array(act_by_class)
im = axes[0,0].imshow(hm, aspect='auto', cmap='YlOrRd')
axes[0,0].set_title('SAE Feature Activation by Class')
axes[0,0].set_xlabel('SAE Latent Feature'); axes[0,0].set_ylabel('Class')
axes[0,0].set_yticks([0,1,2]); axes[0,0].set_yticklabels(['Class 0','Class 1','Class 2'])
plt.colorbar(im, ax=axes[0,0])

# 2. Uncertainty distribution
unc_wrong = unc_cap[preds_cap != yte_cap]
unc_right  = unc_cap[preds_cap == yte_cap]
axes[0,1].hist(unc_right, bins=15, alpha=0.6, color='#1D9E75', density=True, label='Correct')
if len(unc_wrong) > 0:
    axes[0,1].hist(unc_wrong, bins=8,  alpha=0.7, color='#D85A30', density=True, label='Wrong')
axes[0,1].set_title('Ensemble Uncertainty Distribution')
axes[0,1].set_xlabel('Uncertainty'); axes[0,1].legend()

# 3. Coverage-Accuracy curve
axes[1,0].plot(curve_cap['coverage'], curve_cap['accuracy'],
               'o-', color='#534AB7', linewidth=2, markersize=5)
axes[1,0].axhline(acc_cap, linestyle='--', color='#D85A30', label='No abstain')
axes[1,0].scatter([best_row['coverage']], [best_row['accuracy']],
                   color='#1D9E75', s=150, zorder=5, label='Best operating point')
axes[1,0].set_xlabel('Coverage'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].set_title('Selective Prediction Curve'); axes[1,0].legend(fontsize=8)
axes[1,0].grid(alpha=0.3)

# 4. SAE training loss
axes[1,1].plot(sae_cap.losses, color='#534AB7', linewidth=1.8)
axes[1,1].set_title('SAE Training Loss')
axes[1,1].set_xlabel('Step'); axes[1,1].set_ylabel('Loss')
axes[1,1].grid(alpha=0.3)

plt.suptitle('Research Capstone: Interpretable Uncertain ML System', fontsize=12)
plt.tight_layout(); plt.show()

print('\n=== FINAL RESEARCH REPORT ===')
print(f'  Ensemble accuracy (full)        : {acc_cap:.4f}')
print(f'  Selective accuracy (cov≥60%)    : {best_row["accuracy"]:.4f}')
print(f'  Coverage at best operating point: {best_row["coverage"]:.1%}')
print(f'  SAE dead latents                : {n_dead_cap}/{sae_cap.W_enc.shape[1]}')
print(f'  SAE avg sparsity                : {(Z_cap>0).mean():.1%}')

---
## Day 22 Summary

| Concept | Skill Gained |
|---|---|
| L1-SAE | Sparse encoding with L1 penalty, dead latent measurement |
| TopK-SAE | Exactly K active per sample, eliminates dead latents |
| Feature dictionary | SAE latents as interpretable feature directions |
| MC Dropout | T stochastic forward passes, mean + variance estimation |
| Epistemic vs aleatoric | Variance of means vs mean of variances |
| Uncertainty calibration | High uncertainty → wrong predictions |
| Deep ensemble | M diverse models, variance as uncertainty signal |
| Selective prediction | Abstain on high-uncertainty samples |
| Coverage-accuracy curve | Trade coverage for higher accuracy |
| Research capstone | SAE + ensemble + selective prediction pipeline |

---
## Day 23 Preview
- Reinforcement learning: Q-learning deep dive and DQN concepts
- Multi-agent RL: cooperative and competitive settings
- Safe RL: constrained MDPs and reward shaping
- Simulation environments: grid worlds and custom MDPs
- Capstone: train a policy from scratch

In [ ]:
# Bonus: Day 22 in one cell
rng_b = np.random.default_rng(0)
X_b   = rng_b.normal(0, 1, (300, 12))
sae_b = SparseAutoencoder(12, 48, sparsity=0.04, lr=0.01)
for _ in range(150):
    idx = rng_b.integers(0, 300, 64)
    sae_b.step(X_b[idx])
Z_b = sae_b.encode(X_b)
print(f'SAE: dead={((Z_b>0).mean(0)<0.01).sum()}/48  active={( Z_b>0).mean():.1%}')

X_b2, y_b2 = load_iris(return_X_y=True)
Xtr_b, Xte_b, ytr_b, yte_b = train_test_split(X_b2, y_b2, test_size=0.3, random_state=42)
ens_b = DeepEnsemble(5, RandomForestClassifier, n_estimators=30)
ens_b.fit(Xtr_b, ytr_b)
unc_b = ens_b.uncertainty(Xte_b)
pred_b = ens_b.predict(Xte_b)
print(f'Ensemble acc: {accuracy_score(yte_b,pred_b):.4f}  mean_unc: {unc_b.mean():.5f}')
cov_b = unc_b < unc_b.quantile(0.7) if hasattr(unc_b,'quantile') else unc_b < np.quantile(unc_b,0.7)
print(f'Selective acc (cov=70%): {accuracy_score(yte_b[cov_b],pred_b[cov_b]):.4f}')
print('\nDay 22 complete — sparse autoencoders, MC Dropout, deep ensembles, selective prediction!')